In this workbook, I will develop most of the functions and make sure that everything works. Once everything works, I will be using the functions in the modular workbook. The modular workbook is essentials the same as this workbook. I just started this workbook to develop most of the functions i need. I don't like reading the functions in the .py files. I just don't like the way it looks.
    

In [ ]:
#import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns


#import torch
#from torch.utils.data import Dataset, DataLoader
#from torchvision import transforms

#from PIL import Image
#import random
from pathlib import Path
import os


%load_ext autoreload
%autoreload 2

In [ ]:
import importlib
import modular.data_setup as d
import modular.visualization as v
import modular.train_test as t
import modular.engine as e
import modular.model_builder as m


In [ ]:
importlib.reload(v)
importlib.reload(d)
importlib.reload(t)
importlib.reload(e)
importlib.reload(m)

# 0 Become One with the data

## 0.1 Understanding the data

In [ ]:
data_path = Path("data/")
image_path = data_path / "carbonate_1223/PPL-1223"

In [ ]:
# make folders for .py file
!mkdir -p modular

In [ ]:
#%%writefile -a modular/data_setup.py
def walk_through_dir(dir_path):
    """
    Walk through dir_path, ignoring hidden folders,
    and return a DataFrame showing the number of images
    in train, val, and test for each class.
    """
    from pathlib import Path
    import os
    import pandas as pd
    
    # Store counts here
    records = []

    # 1. Walk through directory and print folder contents
    for dirpath, dirnames, filenames in os.walk(dir_path):
        # Ignore hidden folders such as .ipynb_checkpoints
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        # Ignore hidden files
        filenames = [ f for f in filenames if not f.startswith(".")]
        #print(f"There are {len(dirnames)} directories " f"and {len(filenames)} images in '{dirpath}'.")

    # 2. Count images for train / val / test
    for split in ["train", "val", "test"]:
        split_path = os.path.join(dir_path, split)
        # Skip split if it doesn't exist
        if not os.path.isdir(split_path):
            continue
        # Each folder inside split_path is a class
        for class_name in os.listdir(split_path):
            # Ignore hidden folders
            if class_name.startswith("."):
                continue
            class_path = os.path.join(split_path, class_name)
            # Make sure it is actually a directory
            if not os.path.isdir(class_path):
                continue
            # Count files inside the class folder
            num_images = len([
                file
                for file in os.listdir(class_path)
                if not file.startswith(".")
                and os.path.isfile(os.path.join(class_path, file))
            ])
            records.append({
                "class": class_name,
                "split": split,
                "num_images": num_images
            })

    # 3. Convert to DataFrame
    counts_df = pd.DataFrame(records)

    counts_df = (
        counts_df
        .pivot(index="class", columns="split", values="num_images")
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    # Make sure columns exist even if a split is missing
    for split in ["train", "val", "test"]:
        if split not in counts_df.columns:
            counts_df[split] = 0

    # Put columns in desired order
    counts_df = counts_df[
        ["class", "train", "val", "test"]
    ]
    # Add total images for each class
    counts_df["total"] = (
        counts_df["train"]
        + counts_df["val"]
        + counts_df["test"]
    )
   # Percentage of entire dataset represented by each class
    counts_df["percent"] = (counts_df["total"] / counts_df["total"].sum() * 100).round(2)

    # Sort classes numerically: class1, class2, ..., class10, class11, ...
    counts_df["class_number"] = (counts_df["class"].str.extract(r"(\d+)").astype(int))

    counts_df = (counts_df.sort_values("class_number").drop(columns="class_number").reset_index(drop=True))
    return counts_df

In [ ]:
from modular.data_setup import walk_through_dir
counts_df=walk_through_dir(image_path)
counts_df

### Plot the distrubition

In [ ]:
#%%writefile -a modular/visualization.py

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_awesome_barchart(df):
    # Sort values descending by percent for a clean pareto look
    df_sorted = df.sort_values('percent', ascending=False).reset_index(drop=True)
    
    # Set the style background
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create the horizontal barplot
    sns.barplot(
        x='percent', 
        y='class', 
        data=df_sorted, 
        palette="viridis", # A highly readable and modern color palette
        ax=ax
    )
    
    # Add data labels to the end of each bar
    for p in ax.patches:
        width = p.get_width()
        ax.text(
            width + 0.15,                   # Offset slightly to the right of the bar
            p.get_y() + p.get_height() / 2, # Center vertically on the bar
            f'{width:.2f}%', 
            ha='left', 
            va='center', 
            fontsize=10,
            fontweight='bold',
            color='#333333'
        )
        
    # Formatting aesthetics
    ax.set_title('Distribution of Rock Class by Percentage', fontsize=18, fontweight='bold', pad=20)
    ax.set_xlabel('Percentage (%)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Class', fontsize=14, fontweight='bold')
    
    # Remove top and right borders for a cleaner look
    sns.despine(left=True, bottom=True)
    
    # Extend x-axis slightly to make room for the text labels
    ax.set_xlim(0, df_sorted['percent'].max() + 2)
    
    plt.tight_layout()
    plt.show()



In [ ]:
# Execute the function
from modular.visualization import plot_awesome_barchart
plot_awesome_barchart(counts_df)


### Strategies for Handling Class Imbalance

Given the heavy long-tail distribution in this dataset, standard training approaches will likely overfit to the majority classes. To build a robust model, we will combine several approaches across the training pipeline:

#### 1. Data Pipeline & Sampling
*   **Targeted Augmentation:** Apply heavier or specific image augmentations (e.g., rotations, color jitter, cropping) exclusively to the minority classes to artificially increase their variance and representation.
*   **`WeightedRandomSampler`:** Utilize a weighted sampler during the data loading phase to oversample minority classes. This ensures each training batch has a more balanced class distribution.

#### 2. Modeling & Loss Strategy
*   **Transfer Learning:** Fine-tune a pre-trained model rather than training from scratch. Pre-trained weights provide robust feature extractors that help the network generalize even with limited data in rare classes.
*   **Class-Weighted Cross-Entropy:** Apply weights to the loss function that are inversely proportional to class frequencies. This forces the optimizer to treat mistakes on rare classes as more severe than mistakes on majority classes.

#### 3. Evaluation Metrics
*   **Robust Metrics:** Standard accuracy is easily inflated by majority classes. Model selection should be based on **macro F1-score**, **balanced accuracy**, and **per-class recall**.
*   **Confusion Matrix:** Carefully examine the confusion matrix after validation epochs to identify if specific minority classes are being consistently misclassified as majority classes.

#### 4. Dataset Curation
*   **Class Merging:** Re-evaluate the rarest classes (e.g., those with fewer than 100 samples). If they lack distinct geological or feature-level support, consider merging them into visually or logically similar parent classes to improve model stability.

### Summary of Strategies for Imbalanced Datasets

| Approach | What it does | Good for your case? |
| :--- | :--- | :--- |
| **Collect more minority-class images** | Adds real information | Best solution |
| **Merge rare, geologically similar classes** | Reduces impossible classes | Very important |
| **Data augmentation** | Creates variations of minority images | Yes |
| **Weighted loss** | Penalizes minority-class mistakes more | Yes |
| **WeightedRandomSampler** | Shows minority classes more often | Yes |
| **Undersampling majority classes** | Reduces dominance of big classes | Sometimes |
| **Focal loss** | Focuses learning on difficult examples | Worth testing |
| **Transfer learning** | Requires less data than training from scratch | Definitely |
| **Synthetic generation** | Creates artificial minority images | Possible, but risky |
| **Hierarchical classification** | Predict broad group → subclass | Potentially excellent |
| **Better evaluation metrics** | Prevents misleading accuracy | Essential |

## 0.2 Making the dataset

### 0.2.1 Make the full dataset

In [ ]:
#%%writefile -a modular/data_setup.py

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


class RockClassificationDataset(Dataset):

    def __init__(self, root_dir, image_transform=None):
        """
        Args:
            root_dir:
                Path to one split, for example:
                data/carbonate_1223/PPL-1223/train

                Expected structure:

                train/
                    class_1/
                        image1.png
                        image2.png
                    class_2/
                        image3.png
                        image4.png

            image_transform:
                Optional transformations applied to the images.
        """

        self.root_dir = Path(root_dir)
        self.image_transform = image_transform

        # 1. Find all class folders
        self.classes = sorted(
            [folder.name for folder in self.root_dir.iterdir() if folder.is_dir()],
            key=lambda name: int(name.removeprefix("class"))
        )
        # 2. Convert class names into numeric labels
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(self.classes)}
        
        # 3. Collect image paths and their labels
        self.samples = []
        valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
        for class_name in self.classes:
            class_folder = self.root_dir / class_name
            label = self.class_to_idx[class_name]
            for image_path in sorted(class_folder.iterdir()):
                if image_path.suffix.lower() in valid_extensions:
                    self.samples.append(
                        (image_path, label)
                    )
    def __len__(self):
        """
        Return number of images.
        """
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Load one rock image and its class label.
        """
        # 1. Get image path and label
        image_path, label = self.samples[idx]
        # 2. Load image
        image = Image.open(image_path).convert("RGB")
        # 3. Apply transformations
        if self.image_transform:
            image = self.image_transform(image)
        else:
            image = transforms.ToTensor()(image)
        # 4. Return X and y
        return image, label

In [ ]:
from modular.data_setup import RockClassificationDataset
train_dir = "data/carbonate_1223/PPL-1223/train"
val_dir   = "data/carbonate_1223/PPL-1223/val"
test_dir  = "data/carbonate_1223/PPL-1223/test"

train_dataset = RockClassificationDataset(root_dir=train_dir,image_transform=None)

val_dataset = RockClassificationDataset(root_dir=val_dir,image_transform=None)

test_dataset = RockClassificationDataset(root_dir=test_dir,image_transform=None)


### 0.2.2 Make smaller datasets

In [ ]:
#%%writefile -a modular/data_setup.py

from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
import numpy as np
import torch

def make_stratified_subset(dataset: torch.utils.data.Dataset, labels: list, fraction: float=0.2, random_seed=42):
    """
    Create a smaller stratified subset of a PyTorch Dataset.

    Parameters
    ----------
    dataset : torch.utils.data.Dataset
        Original training dataset.

    labels : list or array
        Class label for every image in the dataset.

    fraction : float
        Fraction of dataset to keep.
        Example: 0.2 = keep 20%.

    random_seed : int
        Reproducibility seed.

    Returns
    -------
    torch.utils.data.Subset
    """

    indices = np.arange(len(dataset))

    subset_indices, _ = train_test_split(indices,train_size=fraction,stratify=labels,random_state=random_seed)

    return Subset(dataset, subset_indices)


In [ ]:
# Import data_setup.py
from modular.data_setup import make_stratified_subset
labels = [label for _, label in train_dataset.samples]

train_subset_5_dataset = make_stratified_subset(train_dataset,labels,fraction=0.05,random_seed=42)
train_subset_20_dataset = make_stratified_subset(train_dataset,labels,fraction=0.20,random_seed=42)

## 0.3 Visualize the data

In [ ]:
#%%writefile -a modular/visualization.py
import random
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt


def plot_random_image(image_path, seed=None):
    """
    Randomly selects and displays one image from a directory.

    Args:
        image_path: Path to a directory containing images.
        seed: Optional random seed for reproducible image selection.
    """
    
    # Set random seed if provided
    if seed is not None:
        random.seed(seed)
    image_path = Path(image_path)
    valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    # Get all valid image paths
    image_path_list = [
        path for path in image_path.rglob("*")
        if path.is_file()
        and path.suffix.lower() in valid_extensions
        and ".ipynb_checkpoints" not in path.parts
    ]
    # Make sure images were found
    if len(image_path_list) == 0:
        raise ValueError(f"No images found in: {image_path}")
    # Select random image
    random_image_path = random.choice(image_path_list)
    # Get class from parent folder
    image_class = random_image_path.parent.stem
    # Open image
    img = Image.open(random_image_path)
    # Convert image to NumPy array
    img_as_array = np.asarray(img)
    # Plot
    plt.figure(figsize=(10, 7))
    plt.imshow(img_as_array)
    plt.title(
        f"Image class: {image_class} | "
        f"Image shape: {img_as_array.shape} "
        f"-> [height, width, color_channels]"
    )

    plt.axis("off")
    plt.show()

In [ ]:
from modular.visualization import plot_random_image
plot_random_image("data/carbonate_1223/PPL-1223/train", seed=None)

### Dataset Audit: DeepCarbonate Quality & Consistency Review

An inspection of the **DeepCarbonate** dataset identified critical inconsistencies between the folder labels, image filenames, and geological classifications across all three optical modes (**PPL**, **XPL**, and **reflected light**).

---

#### ⚠️ Labeling & Filename Discrepancies

* **Class 1 Naming Mismatch:** The training images in `class1` are named `Vug.*`, whereas the validation and test images are named `Arenaceous.*`. However, the supplied class map identifies `class1` as **Microbial mudstone**.
* **Overlapping Filenames:** `class22` (identified as **Pore**) also contains images named `Vug.*`.
* **Index Non-Uniqueness:** In the **PPL training split**, **1,275 filenames** occur simultaneously in both `class1` and `class22`. 
    > 🔍 *Note: File-content and perceptual comparisons show that these are completely different images rather than exact or near duplicates. The filename index is therefore not globally unique.*

#### 🧠 Conceptual Inconsistencies

There is a fundamental mismatch in describing all 22 targets strictly as lithological classes:
* **Feature vs. Rock Type:** A **pore** is not a rock type; it is a petrographic or reservoir feature. 
* **Other Affected Categories:** Classes such as **cemented fracture** and **stylolite** represent diagenetic features rather than complete rock lithologies.

> **Revised Definition:** DeepCarbonate is more accurately described as a **petrographic image-category dataset** that combines lithologies, depositional textures, and reservoir features.

---

#### 🛠️ Actionable Pipeline Recommendations

1. **Folder-Based Pipelines:** Until the `class1` naming discrepancy is verified, class assignments **must be derived strictly from folder placement** rather than individual filenames (e.g., using PyTorch's `datasets.ImageFolder`). Do not use filenames to determine an image's class.
2. **Manual Visual Audits:** Representative images from `class1` should be manually reviewed to verify the actual structural characteristics.
3. **Model Architecture Adjustments:**
    * *To Benchmark:* Models reproducing the original benchmark can retain all 22 categories as-is.
    * *For Strict Rock Classifiers:* A pure rock-lithology classifier should either **exclude feature-based categories** entirely or treat them as separate **multi-label outputs**.


# 1. Transformations

## 1.0 Make the transformation functions

In [ ]:
#%%writefile -a modular/data_setup.py
import torch
def calculate_mean_std(dataloader):
    """
    Calculate the RGB mean and standard deviation using float64.
    """
    channel_sum = torch.zeros(3, dtype=torch.float64)
    channel_sum_squared = torch.zeros(3, dtype=torch.float64)
    num_pixels = 0

    for images, _ in dataloader:
        images = images.to(dtype=torch.float64)

        channel_sum += images.sum(dim=(0, 2, 3))
        channel_sum_squared += images.square().sum(dim=(0, 2, 3))

        num_pixels += (
            images.size(0)
            * images.size(2)
            * images.size(3)
        )

    if num_pixels == 0:
        raise ValueError("Cannot calculate statistics from an empty dataset.")

    mean = channel_sum / num_pixels

    variance = (
        channel_sum_squared / num_pixels
        - mean.square()
    ).clamp_min(0)

    std = torch.sqrt(variance)

    return mean, std
    

In [ ]:
# input u can put into for mean and std if u are important a model 
mean_imagenet=[0.485, 0.456, 0.406]
std_imagenet  = [0.229, 0.224, 0.225]

In [ ]:
#%%writefile -a modular/data_setup.py
from torchvision import transforms
def remove_bottom_annotation(image, pixels):
    """
    this function can be used to remove the bottom part. It removes the ruler.
    """
    width, height = image.size
    if not 0 <= pixels < height:
        raise ValueError(f"pixels must be between 0 and {height - 1}, got {pixels}")
    return image.crop((0, 0, width, height - pixels))

In [ ]:
#%%writefile -a modular/data_setup.py
from torchvision import transforms
from torch.utils.data import DataLoader,Dataset, Subset
import copy

def create_transformation(
    train_dataset: Dataset, 
    IMG_SIZE:int =224,
    resize:int =256,
    pixels: int = 90,
    normalization: str = "mydataset",
    mean_ext=None,
    std_ext=None,

    batch_size:int =32,
    num_workers:int = 4,
    #RandomResizedCrop_scale_min: float =0.8,
    #RandomResizedCrop_scale_max: float =1.0,
    #RandomResizedCrop_ratio_min: float =0.9,
    #RandomResizedCrop_ratio_max: float =1.1,
    RandomHorizontalFlip_p: float = 0.5,
    RandomVerticalFlip_p:float = 0.0,  # never flip for now. #If bedding direction, sedimentary structures, laminations, or other directional geological features matter, test whether vertical flip helps or hurts.
    #RandomRotation_degrees:int = 0,  # keep this as zeros as it creates issues
    brightness:float = 0.15,
    contrast:float = 0.15,
    saturation:float = 0.1,
    hue:float = 0.03,
):
    """
Creates training and validation/test image transformations for a rock classification dataset.

Args:
    train_dataset: A PyTorch Dataset or Subset used to calculate normalization
        statistics and determine the training dataset size.
    IMG_SIZE: Final image size passed to the model. Default is 224.
    resize: Intermediate resize value used before CenterCrop for statistics
        calculation and validation/test preprocessing. Default is 256.
    normalization: Normalization method to use. Should be either:
        "mydataset" to calculate mean and standard deviation from train_dataset,
        or "external" to use externally supplied mean and standard deviation values.
    mean_ext: External RGB mean values used when normalization="external".
    std_ext: External RGB standard deviation values used when normalization="external".
    batch_size: Batch size used when calculating dataset mean and standard deviation.
    num_workers: Number of DataLoader worker processes used when calculating
        dataset statistics.
    RandomResizedCrop_scale_min: Minimum crop area scale for RandomResizedCrop.
    RandomResizedCrop_scale_max: Maximum crop area scale for RandomResizedCrop.
    RandomResizedCrop_ratio_min: Minimum aspect ratio for RandomResizedCrop.
    RandomResizedCrop_ratio_max: Maximum aspect ratio for RandomResizedCrop.
    RandomHorizontalFlip_p: Probability of applying a horizontal flip.
    RandomVerticalFlip_p: Probability of applying a vertical flip.
    RandomRotation_degrees: Maximum rotation angle in degrees. Images may be
        rotated between -degrees and +degrees.
    brightness: Maximum brightness variation used by ColorJitter.
    contrast: Maximum contrast variation used by ColorJitter.
    saturation: Maximum saturation variation used by ColorJitter.
    hue: Maximum hue variation used by ColorJitter.
    pixels: Number of pixels removed from the bottom of each image
    to remove the scale bar/annotation. Default is 80.

Returns:
    train_transform: PyTorch transformation pipeline containing training
        augmentations and normalization.
    val_test_transform: Deterministic transformation pipeline for validation
        and test images.
    mean: RGB mean values used for normalization.
    std: RGB standard deviation values used for normalization.
    dataset_len: Number of samples in the supplied training dataset.

Example usage:
    train_transform, val_test_transform, mean, std, dataset_len = create_transformation(
        train_dataset=train_subset_20_dataset,
        IMG_SIZE=224,
        resize=256,
        normalization="mydataset"
    )

    # Using ImageNet statistics for a pretrained model:
    train_transform, val_test_transform, mean, std, dataset_len = create_transformation(
        train_dataset=train_subset_20_dataset,
        normalization="external",
        mean_ext=[0.485, 0.456, 0.406],
        std_ext=[0.229, 0.224, 0.225]
    )
"""
    # 1. Write separate transforms for train and test data
    from torchvision import transforms
    from torch.utils.data import DataLoader
    
    #IMG_SIZE = IMG_SIZE # can change this for different experiments
    # ============================================================
    # 1. CALCULATE TRAINING DATASET STATISTICS
    # ============================================================
    #------------------------------------------------------------------------------------------
    # Write a transform to create my own statistics for the standardization
    assert resize >= IMG_SIZE, "resize must be greater than or equal to IMG_SIZE"
    stats_transform = transforms.Compose([
        transforms.Lambda(lambda img: remove_bottom_annotation(img, pixels=pixels)),
        transforms.Resize(resize),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor()
    ])
    
    if isinstance(train_dataset, Subset):
        # Copy the original dataset underlying the subset
        train_dataset_stats_full = copy.copy(train_dataset.dataset)
        # Give the copy the deterministic statistics transform
        train_dataset_stats_full.image_transform = stats_transform
        # Keep exactly the same subset indices
        train_dataset_stats = Subset(train_dataset_stats_full,train_dataset.indices)
    else:
        # Copy the full dataset
        train_dataset_stats = copy.copy(train_dataset)
    
        # Change only the transform on the copy
        train_dataset_stats.image_transform = stats_transform
    
    # Make dataloader for stats if necessary  Then calculate the mean and standard deviation. Have the option to ue my dataset, or an external one. 
    if normalization == "mydataset":
        # Calculate statistics from your selected training dataset
        stats_loader = DataLoader(train_dataset_stats,batch_size=batch_size,shuffle=False,num_workers=num_workers)
        mean, std = calculate_mean_std(stats_loader)
        mean = mean.tolist()
        std = std.tolist()
    elif normalization == "external":
    
        if mean_ext is None or std_ext is None:
            raise ValueError("mean_ext and std_ext must be provided when normalization='external'")
        mean = mean_ext
        std = std_ext
    else:
        raise ValueError("normalization must be either 'mydataset' or 'external'")
    
    #######################################
    #2. TRAINING TRANSFORM
    ########################################
    #------------------------------------------------------------------------------------------
    train_transform = transforms.Compose([  
        transforms.Lambda(lambda img: remove_bottom_annotation(img, pixels=pixels)),
        #transforms.RandomRotation(degrees=RandomRotation_degrees),# Small rotation
        #transforms.RandomResizedCrop(IMG_SIZE, scale=(RandomResizedCrop_scale_min, RandomResizedCrop_scale_max),ratio=(RandomResizedCrop_ratio_min, RandomResizedCrop_ratio_max)),
        transforms.Resize(resize),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=RandomHorizontalFlip_p),# Orientation augmentation
        transforms.RandomVerticalFlip(p=RandomVerticalFlip_p), #If bedding direction, sedimentary structures, laminations, or other directional geological features matter, test whether vertical flip helps or hurts.
        transforms.ColorJitter(brightness=brightness,contrast=contrast,saturation=saturation,hue=hue),# Lighting/camera variation
        #transforms.TrivialAugmentWide(num_magnitude_bins=31), can experiment with this later
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
     ]) 
    dataset_len = len(train_dataset)
    # ============================================================
    # 3. VALIDATION / TEST TRANSFORM
    # ============================================================
    val_test_transform = transforms.Compose([
        transforms.Lambda(lambda img: remove_bottom_annotation(img, pixels=pixels)),
        transforms.Resize(resize),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean,std=std)
    ])
    return train_transform, val_test_transform, mean, std,dataset_len

### Testing the transforms function

In [ ]:
from modular.data_setup import create_transformation

train_transform, val_test_transform, mean, std,dataset_len = create_transformation(
    train_dataset=train_dataset,normalization = "mydataset", pixels=90
)
print(f"train transform is \n{train_transform}")
print(f"val and test transform is \n{val_test_transform}")
print(f"mean is \n{mean}")
print(f"std \n{std}")
print(f"length of data is  \n{dataset_len}")

In [ ]:
train_transform_5, val_test_transform_5, mean_5, std_5,dataset_len_5 = create_transformation(train_dataset=train_subset_5_dataset,normalization = "mydataset", pixels=90)
print(f"train transform is \n{train_transform_5}")
print(f"val and test transform is \n{val_test_transform_5}")
print(f"mean is \n{mean_5}")
print(f"std is \n{std_5}")
print(f"length of data is  \n{dataset_len_5}")


In [ ]:
train_transform_20, val_test_transform_20, mean_20, std_20,dataset_len_20 = create_transformation(
    train_dataset=train_subset_20_dataset,normalization = "mydataset", pixels=90
)
print(f"train transform is \n{train_transform_20}")
print(f"val and test transform is \n{val_test_transform_20}")
print(f"mean is \n{mean_20}")
print(f"std \n{std_20}")
print(f"length of data is  \n{dataset_len_20}")


#### other things to try

| Transformation | Use? | Reason |
|---|---|---|
| **Resize** | ✅ | Required for batching/pretrained network |
| **RandomResizedCrop** | ✅ | Helps model not depend on exact framing |
| **Horizontal flip** | ✅ | Rock type usually shouldn't change if mirrored |
| **Vertical flip** | ⚠️ | Depends on whether geological orientation matters |
| **Rotation ±15°** | ✅ | Camera orientation shouldn't control classification |
| **Brightness** | ✅ | Handles illumination variation |
| **Contrast** | ✅ | Handles shadows/exposure |
| **Saturation** | ✅ mild | Handles camera/weather variation |
| **Hue** | ⚠️ very mild | Rock color can be diagnostic |
| **Gaussian blur** | ⚠️ | Can destroy grain/texture information |
| **Random erasing** | ⚠️ | Could hide diagnostic geological features |
| **Perspective distortion** | ❌ initially | Often physically unrealistic |
| **Huge rotation/distortion** | ❌ | May create unrealistic samples |

#### Experiment 1 — Minimal
* RandomResizedCrop
* HorizontalFlip
* Normalize
  
#### Experiment 1 — Minimal
* RandomResizedCrop
* verticalFlip
* Normalize

#### Experiment 2 — Recommended
* RandomResizedCrop
* HorizontalFlip
* Rotation ±15°
* Mild ColorJitter
* Normalize

#### Experiment 3 — Stronger
* RandomResizedCrop
* HorizontalFlip
* VerticalFlip
* Rotation ±20°
* ColorJitter
* RandomAffine
* Normalize

## 1.1 Veiw results of transformation

In [ ]:
#%%writefile -a modular/visualization.py
import random
import torch
import matplotlib.pyplot as plt
from PIL import Image


def plot_transformed_images(
    image_paths,
    transform,
    mean,
    std,
    n: int = 3,
    seed = None
):
    """
    Plots original rock images beside their transformed versions.

    Randomly selects n image paths, applies the supplied PyTorch
    transformation, unstandardizes the transformed image for
    visualization, and plots the original and transformed images
    side by side.

    Args:
        image_paths: List of image paths.
        transform: PyTorch transformation pipeline to apply to each image.
        mean: RGB mean values used for image normalization.
        std: RGB standard deviation values used for image normalization.
        n: Number of random images to display. Default is 3.
        seed: Random seed used for reproducible image selection.
            Default is 42.

    Example usage:
        plot_transformed_images(
            image_paths=image_path_list,
            transform=train_transform,
            mean=mean,
            std=std,
            n=3,
            seed=42
        )
    """
    # Set random seeds
    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)

    # Make sure n is not larger than number of available images
    n = min(n, len(image_paths))
    # Randomly select images
    random_image_paths = random.sample(image_paths, k=n)
    # Convert mean/std into shape [C, 1, 1]
    # so they can be applied to an image tensor [C, H, W]
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    for image_path in random_image_paths:
        # Load image
        with Image.open(image_path).convert("RGB") as image:
            # ----------------------------------------
            # Apply transformation
            # ----------------------------------------
            transformed_image = transform(image)
            # ----------------------------------------
            # Undo normalization for visualization
            # x_original = x_normalized * std + mean<abs
            # ----------------------------------------
            transformed_image_display = (transformed_image * std_tensor + mean_tensor)
            # Keep pixel values between 0 and 1
            transformed_image_display = transformed_image_display.clamp(0, 1)
            # Convert [C, H, W] -> [H, W, C]
            transformed_image_display = transformed_image_display.permute(1, 2, 0)
            # ----------------------------------------
            # Plot
            # ----------------------------------------
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            # Original
            ax[0].imshow(image)
            ax[0].set_title(f"Original\nSize: {image.size}")
            ax[0].axis("off")
            # Transformed
            ax[1].imshow(transformed_image_display,interpolation="none")
            ax[1].set_title(
                f"Transformed\n"
                f"Size: {transformed_image.shape[1]} × "
                f"{transformed_image.shape[2]}"
            )
            ax[1].axis("off")
            # Class name comes from parent folder
            fig.suptitle(f"Class: {image_path.parent.name}",fontsize=16)
            plt.tight_layout()
            plt.show()

In [ ]:
image_path_list_5 = [
    train_subset_5_dataset.dataset.samples[i][0]
    for i in train_subset_5_dataset.indices
]

In [ ]:
from modular.visualization import plot_transformed_images

plot_transformed_images(
    image_paths=image_path_list_5,
    transform=train_transform_5,
    mean=mean_5,
    std=std_5,
    n=5,
    seed=None
)

# 2. Loading the dataset

In [ ]:
# Run this inside a notebook cell to delete all hidden checkpoints in your data folder
!find data/ -type d -name ".ipynb_checkpoints" -exec rm -rf {} +

## 2.1 Give each subset its OWN transformed dataset

In [ ]:
#%%writefile -a modular/data_setup.py
import copy
from torch.utils.data import Subset

def subset_with_transform(subset, transform):
    """
    Create an independent copy of a Subset's underlying dataset
    and attach a new transform while preserving the same subset indices.
    """
    
    dataset_copy = copy.copy(subset.dataset)
    dataset_copy.image_transform = transform
    
    return Subset(
        dataset_copy,
        subset.indices
    )

In [ ]:
from modular.data_setup import subset_with_transform
train_subset_5_dataset = subset_with_transform(
    train_subset_5_dataset,
    train_transform_5
)
train_subset_20_dataset = subset_with_transform(
    train_subset_20_dataset,
    train_transform_20
)


## 5 % of the data

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_dataloader_5 = DataLoader(
    train_subset_5_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)
val_dataset_5 = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=val_test_transform_5
)
val_dataloader_5 = DataLoader(
    val_dataset_5,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

In [ ]:
print(train_subset_5_dataset.dataset.image_transform)
print(val_dataset_5.image_transform)

## 20 % of the data

In [ ]:

from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_dataloader_20 = DataLoader(
    train_subset_20_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_dataset_20 = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=val_test_transform_20
)

val_dataloader_20 = DataLoader(
    val_dataset_20,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

In [ ]:
print(train_subset_20_dataset.dataset.image_transform)
print(val_dataset_20.image_transform)

## All of the data

In [ ]:

from torch.utils.data import DataLoader
BATCH_SIZE=32
train_dataset.image_transform = train_transform
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_dataset = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=val_test_transform
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

In [ ]:
print(train_dataset.image_transform)
print(val_dataset.image_transform)

# 3. Pre Models Needs

## 3.0 Check GPU

In [ ]:
import torch

In [ ]:
!nvidia-smi

In [ ]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch CUDA version:", torch.version.cuda)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

## 3.1 Impot torch metrics

In [ ]:
from torchmetrics import MetricCollection 
from torchmetrics.classification import (MulticlassAccuracy,MulticlassF1Score,MulticlassPrecision,MulticlassRecall)

In [ ]:
class_names = train_dataset.classes
metrics = MetricCollection({
    "accuracy": MulticlassAccuracy(num_classes=len(class_names),average="micro"),
    "f1": MulticlassF1Score(num_classes=len(class_names),average="macro"),
    "precision": MulticlassPrecision(num_classes=len(class_names),average="macro"),
    "recall": MulticlassRecall(num_classes=len(class_names),average="macro")
})

In [ ]:
train_metrics = metrics.clone(prefix="train_").to(device)
test_metrics = metrics.clone(prefix="test_").to(device)

#### micro vs Marco

When evaluating a multiclass model on an imbalanced dataset, the choice between micro and macro averaging dictates whether you are giving equal weight to every image or every class.

* **Micro-Averaging (Image-focused):** Pools all predictions together before calculating the metric. Because it weighs every image equally, massive classes will dominate the final score. It answers: *"What percentage of total images were correct?"*
* **Macro-Averaging (Class-focused):** Calculates the metric for each class individually, then averages those class scores. It treats all classes equally, regardless of how many images they have. It answers: *"How well does the model perform across all classes on average?"*

**The Key Takeaway:** If you have an imbalanced dataset, a high Micro Accuracy can hide terrible performance on minority classes. Using Macro metrics (like Macro F1) exposes those weaknesses and provides a truer picture of the model's overall capability. 

If you run these metrics on your dashboard and see the following results:

| Metric | Score | Interpretation |
| :--- | :--- | :--- |
| **Accuracy (Micro)** | 91% | The model gets the vast majority of *total images* right (likely just guessing the most common rocks). |
| **Precision (Macro)** | 68% | When it predicts a specific rock type, it is correct about 68% of the time, averaged across all types. |
| **Recall (Macro)** | 61% | It successfully finds 61% of the actual instances of a given rock type, averaged across all types. |
| **F1 Score (Macro)** | **63%** | **The most informative metric here.** The large gap between the 91% Accuracy and 63% Macro F1 proves the model is struggling significantly with the minority rock classes. |

## 3.2 train function

In [ ]:
#%%writefile -a modular/train_test.py
from tqdm.auto import tqdm
from timeit import default_timer as timer
import torch
from torch import nn
from torch.optim import Optimizer
from torch.utils.data import DataLoader

def train_step(model: torch.nn.Module, 
               train_dataloader: torch.utils.data.DataLoader, 
               loss_fn: torch.nn.Module, 
               optimizer: torch.optim.Optimizer,
               metrics,
               device: torch.device,
              ):
    train_loss=0
    # put model into training mode
    model.train()

    # add a loop to loop through the training batches
    for batch, (X,y) in enumerate (train_dataloader):
        # Put data on target device
        X=X.to(device)
        y=y.to(device)
        
        # 1. Forward pass
        y_logits=model(X)   # result of model
        y_pred_probs = torch.log_softmax(y_logits, dim=1) # dim=1 calculates log-probabilities across rows. # Use logsoftmax function on our model logits to turn them into prediction probabilities. the rows of LogSoftmax outputs will not add up to 1. Instead, if you take the exponent (\(e^{x}\)) of each number in a row, those exponents will add up 1
        y_preds=torch.argmax(y_pred_probs, dim=1)  # find the predicted labels
            
        #2. Calculate Loss
        loss =loss_fn(y_logits, y)
        train_loss+= loss.item()  # accumilate train loass
        metrics.update(y_preds, y)
        
        #3. optimizer. Have to zero it for each epoch
        optimizer.zero_grad()
        
        #4. Back propagration
        loss.backward()
        
        #5. Step the optimizer
        optimizer.step()
    
    
    # divide total train loss and acc by length of train datest_dataloader
    train_loss /=len(train_dataloader) # if u want you can use something like train_loss_avg = train_loss / len(train_dataloader)
    
    # Calculate metrics for entire epoch
    train_metric_results = metrics.compute()

    # Reset for next epoch
    metrics.reset()


    return train_loss, train_metric_results

## 3.3 Step function

In [ ]:
#%%writefile -a modular/train_test.py

def test_step(model: torch.nn.Module, 
               test_dataloader: torch.utils.data.DataLoader, 
               loss_fn: torch.nn.Module, 
               metrics,
               device: torch.device ,
              ):
    test_loss=0 
    model.eval() # turns off different settings in the model not needed for evaluation

    with torch.inference_mode(): # turns off graident tracking and a couple of other things
        for X,y in test_dataloader:
            X=X.to(device)
            y=y.to(device)
            
            #1. Do forward pass
            test_logits=model(X)   #
            test_probs = torch.log_softmax(test_logits, dim=1) 
            test_pred=torch.argmax(test_probs, dim=1) 
            
            #2. Calculate the loss
            test_loss += loss_fn(test_logits, y).item()
            #3. Calculate the acc
            metrics.update(test_pred, y)
            
        # calculate the test loss average per test
        test_loss/=len(test_dataloader)
        
        # Calculate metrics over entire validation dataset
        test_metric_results = metrics.compute()
    
        # Reset before next epoch
        metrics.reset()      

    return test_loss, test_metric_results

## 3.4 Combine both step and train function

#### Packaging Training and Evaluation into a `train()` Function

Now we need a way to put our `train_step()` and `test_step()` functions together. To do so, we'll package them up in a `train()` function. This function will handle both training the model and evaluating it over a specified number of epochs.

##### Function Workflow
The `train()` function will perform the following steps:
1. **Accept Parameters**: Takes in a model, a training `DataLoader`, a test `DataLoader`, an optimizer, a loss function, and the number of epochs to run.
2. **Initialize Results Tracker**: Creates an empty results dictionary with keys for `train_loss`, `train_acc`, `test_loss`, and `test_acc`. We will populate these lists as training progresses.
3. **Execute Epoch Loop**: Loops through the training and test step functions for the specified number of epochs.
4. **Log Progress**: Prints out metrics showing what is happening at the end of each epoch.
5. **Update and Return Metrics**: Updates the results dictionary with the latest metrics every epoch and returns the filled dictionary.

To keep track of the number of epochs we've been through, we will import `tqdm` from `tqdm.auto`. `tqdm` is one of the most popular progress bar libraries for Python, and `tqdm.auto` automatically decides what kind of progress bar is best for your computing environment (e.g., Jupyter Notebook vs. a standard Python script).


In [ ]:
#%%writefile -a modular/train_test.py

from torch import nn
from tqdm.auto import tqdm
from timeit import default_timer as timer
import wandb

# 1. Take in various parameters required for training and test steps
def train_test(
        model: torch.nn.Module,
        train_dataloader: torch.utils.data.DataLoader,
        test_dataloader: torch.utils.data.DataLoader,
        optimizer: torch.optim.Optimizer,
        train_metrics,
        test_metrics,
        device,
        loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
        epochs: int = 5
):
    
    # 2. Create empty results dictionary
    results = {
        "train_loss": [],
        "train_acc": [],
        "train_f1": [],
        "train_precision": [],
        "train_recall": [],
        
        "test_loss": [],
        "test_acc": [],
        "test_f1": [],
        "test_precision": [],
        "test_recall": [],

        "epoch_time": []
}
    
    # 3. Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        
        epoch_start = timer()
        
        train_loss, train_results = train_step(model=model,
                                           train_dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           metrics=train_metrics,
                                            device=device)
                                           
        test_loss, test_results = test_step(model=model,
                                        test_dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        metrics=test_metrics,
                                        device=device)

        epoch_end = timer()
        epoch_time = epoch_end - epoch_start
        
        # 4. Print epoch results
        print(
            f"Epoch: {epoch+1} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_results['train_accuracy'].item()*100:.2f}% | "
            f"Train F1: {train_results['train_f1'].item():.4f} | "
            f"train_precision: {train_results['train_precision'].item():.4f} | "
            f"train_recall: {train_results['train_recall'].item():.4f} | "

            f"Test loss: {test_loss:.4f} | "
            f"Test acc: {test_results['test_accuracy'].item()*100:.2f}% | "
            f"Test F1: {test_results['test_f1'].item():.4f}"
            f"test_precision: {test_results['test_precision'].item():.4f}"
            f"test_recall: {test_results['test_recall'].item():.4f}"
        )

        # 4. Save training results
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_results["train_accuracy"].item())
        results["train_f1"].append(train_results["train_f1"].item())
        results["train_precision"].append(train_results["train_precision"].item())
        results["train_recall"].append(train_results["train_recall"].item())

        # 5. Save test results
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_results["test_accuracy"].item())
        results["test_f1"].append(test_results["test_f1"].item())
        results["test_precision"].append(test_results["test_precision"].item())
        results["test_recall"].append(test_results["test_recall"].item())

        results["epoch_time"].append(epoch_time)
        
        # 6. Log to wandb
        wandb.log({
            "epoch": epoch + 1,
        
            "train/loss": train_loss,
            "train/accuracy": train_results["train_accuracy"].item(),
            "train/f1": train_results["train_f1"].item(),
            "train/precision": train_results["train_precision"].item(),
            "train/recall": train_results["train_recall"].item(),
        
            "val/loss": test_loss,
            "val/accuracy": test_results["test_accuracy"].item(),
            "val/f1": test_results["test_f1"].item(),
            "val/precision": test_results["test_precision"].item(),
            "val/recall": test_results["test_recall"].item(),
        
            "epoch_time": epoch_time
        })

    return results

## 3.5 Make the loss curve of function

In [ ]:
#%%writefile -a modular/visualization.py
import matplotlib.pyplot as plt
import numpy as np


def plot_training_metrics(results, model_name):

    epochs = range(1, len(results["train_loss"]) + 1)

    metrics = [("loss", "Loss"),("acc", "Accuracy"),("f1", "F1 Score"),("precision", "Precision"),("recall", "Recall")]
    # -------------------------------------------------
    # Create 2 x 3 subplot figure
    # -------------------------------------------------
    fig, axes = plt.subplots(2,3,figsize=(18, 10))
    axes = axes.flatten()
    # -------------------------------------------------
    # Training / Validation metrics
    # -------------------------------------------------
    for ax, (key, title) in zip(axes, metrics):
        ax.plot(epochs,results[f"train_{key}"],marker="o",label=f"Train {title}")
        ax.plot(epochs,results[f"test_{key}"],marker="o",label=f"Validation {title}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.set_title(f"Training vs Validation {title}")
        ax.legend()
        ax.grid(alpha=0.3)

    # -------------------------------------------------
    # Cumulative training time
    # -------------------------------------------------
    cumulative_time = np.cumsum(results["epoch_time"])
    ax = axes[5]
    ax.plot(epochs,cumulative_time,marker="o")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cumulative Time (seconds)")
    ax.set_title("Cumulative Training Time")
    ax.grid(alpha=0.3)

    # -------------------------------------------------
    # Overall figure title
    # -------------------------------------------------
    fig.suptitle(
        f"{model_name} — Training Performance",
        fontsize=20,
        fontweight="bold"
    )

    # -------------------------------------------------
    # Adjust spacing
    # -------------------------------------------------
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    plt.show()

## 3.6 Make the Diagnostic function Plot

In [ ]:
#%%writefile -a modular/visualization.py
import matplotlib.pyplot as plt
import numpy as np
import torch

from torchmetrics import ConfusionMatrix
from torchmetrics.classification import (
    MulticlassF1Score,
    MulticlassRecall
)

from mlxtend.plotting import plot_confusion_matrix


def plot_model_diagnostics(
    model,
    train_dataloader,
    test_dataloader,
    class_names,
    model_name,
    device
):

    # =========================================================
    # 1. GET MODEL PREDICTIONS
    # =========================================================
    model.eval()

    y_preds = []
    y_true = []

    with torch.inference_mode():

        for X, y in test_dataloader:

            X = X.to(device)
            y = y.to(device)

            # Forward pass
            y_logits = model(X)

            # Predicted class
            y_pred = torch.argmax(y_logits, dim=1)

            # Store predictions and true labels
            y_preds.append(y_pred.cpu())
            y_true.append(y.cpu())


    # Combine all batches
    y_pred_tensor = torch.cat(y_preds)
    y_true_tensor = torch.cat(y_true)


    # =========================================================
    # 2. CONFUSION MATRIX
    # =========================================================
    confmat = ConfusionMatrix(
        num_classes=len(class_names),
        task="multiclass"
    )

    confmat_tensor = confmat(
        preds=y_pred_tensor,
        target=y_true_tensor
    )


    # =========================================================
    # 3. F1 SCORE FOR EACH CLASS
    # =========================================================
    f1_metric = MulticlassF1Score(
        num_classes=len(class_names),
        average=None
    )

    f1_scores = f1_metric(
        y_pred_tensor,
        y_true_tensor
    ).numpy()


    # =========================================================
    # 4. RECALL FOR EACH CLASS
    # =========================================================
    recall_metric = MulticlassRecall(
        num_classes=len(class_names),
        average=None
    )

    recall_scores = recall_metric(
        y_pred_tensor,
        y_true_tensor
    ).numpy()


    # =========================================================
    # 5. GET TRAINING CLASS COUNTS
    # =========================================================
    train_labels = [
        label
        for _, label in train_dataloader.dataset
    ]

    train_counts = np.bincount(
        train_labels,
        minlength=len(class_names)
    )


    # =========================================================
    # 6. CREATE ONE FIGURE WITH 5 SUBPLOTS
    # =========================================================
    fig, axes = plt.subplots(2,3,figsize=(20, 12))

    # Main title
    fig.suptitle(
        f"{model_name} — Model Diagnostics",
        fontsize=20,
        fontweight="bold"
    )


    # =========================================================
    # PLOT 1 — CONFUSION MATRIX
    # =========================================================
    plot_confusion_matrix(
        conf_mat=confmat_tensor.numpy(),
        class_names=class_names,
        figure=fig,
        axis=axes[0, 0]
    )

    axes[0, 0].set_title(
        "Confusion Matrix"
    )


    # =========================================================
    # PLOT 2 — F1 SCORE BY CLASS
    # =========================================================
    axes[0, 1].barh(
        class_names,
        f1_scores
    )

    axes[0, 1].set_title(
        "F1 Score by Class"
    )

    axes[0, 1].set_xlabel(
        "F1 Score"
    )

    axes[0, 1].set_xlim(
        0,
        1
    )

    axes[0, 1].grid(
        alpha=0.3
    )


    # =========================================================
    # PLOT 3 — RECALL BY CLASS
    # =========================================================
    axes[0, 2].barh(
        class_names,
        recall_scores
    )

    axes[0, 2].set_title(
        "Recall by Class"
    )

    axes[0, 2].set_xlabel(
        "Recall"
    )

    axes[0, 2].set_xlim(
        0,
        1
    )

    axes[0, 2].grid(
        alpha=0.3
    )


    # =========================================================
    # PLOT 4 — TRAINING CLASS DISTRIBUTION
    # =========================================================
    axes[1, 0].barh(
        class_names,
        train_counts
    )

    axes[1, 0].set_title(
        "Training Class Distribution"
    )

    axes[1, 0].set_xlabel(
        "Number of Images"
    )

    axes[1, 0].grid(
        alpha=0.3
    )


    # =========================================================
    # PLOT 5 — TRAINING IMAGES VS F1
    # =========================================================
    axes[1, 1].scatter(
        train_counts,
        f1_scores
    )

    for i, class_name in enumerate(class_names):

        axes[1, 1].annotate(
            class_name,
            (
                train_counts[i],
                f1_scores[i]
            )
        )

    axes[1, 1].set_title(
        "Training Images vs F1 Score"
    )

    axes[1, 1].set_xlabel(
        "Number of Training Images"
    )

    axes[1, 1].set_ylabel(
        "F1 Score"
    )

    axes[1, 1].set_ylim(
        0,
        1
    )

    axes[1, 1].grid(
        alpha=0.3
    )


    # =========================================================
    # REMOVE UNUSED 6TH SUBPLOT
    # =========================================================
    axes[1, 2].axis("off")


    plt.tight_layout()
    plt.show()

## 3.7 Function to summary dataframe experiment

In [ ]:
#%%writefile modular/utility.py
def summarize_experiment(
    results_df,
    model,
    experiment_name,
    model_name,
    img_size,
    loss_fn,
    notes=None
):
    """
    Summarizes the final-epoch performance of a model training experiment.

    Records the training and validation metrics from the last epoch,
    along with training-time statistics and important experiment settings.

    The returned dictionary can be combined with summaries from other
    experiments to create an experiment comparison dataframe.
    """

    summary = {
        "experiment": experiment_name,
        "model": model_name,
        "img_size": img_size,
        "loss_fn": loss_fn,

        "num_parameters": sum(p.numel() for p in model.parameters()),
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),

        "train_loss": results_df["train_loss"].iloc[-1].round(3),
        "train_acc": results_df["train_acc"].iloc[-1].round(3),
        "train_f1": results_df["train_f1"].iloc[-1].round(3),
        "train_precision": results_df["train_precision"].iloc[-1].round(3),
        "train_recall": results_df["train_recall"].iloc[-1].round(3),

        "val_loss": results_df["test_loss"].iloc[-1].round(3),
        "val_acc": results_df["test_acc"].iloc[-1].round(3),
        "val_f1": results_df["test_f1"].iloc[-1].round(3),
        "val_precision": results_df["test_precision"].iloc[-1].round(3),
        "val_recall": results_df["test_recall"].iloc[-1].round(3),

        "avg_epoch_time": results_df["epoch_time"].mean().round(0),
        "total_training_time": results_df["epoch_time"].sum().round(0),

        "notes": notes
    }

    return summary

# 4. Models

## 4.0 model_0_0_tinyVGG 

In [ ]:

from torch import nn
class TinyVGG(nn.Module):
    """
    Model architecture copying TinyVGG from: 
    https://poloclub.github.io/cnn-explainer/
    """
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, 
                      out_channels=hidden_units, 
                      kernel_size=3, # how big is the square that's going over the image?
                      stride=1, # default
                      padding=0), # options = "valid" (no padding) or "same" (output has same shape as input) or int for specific number 
            nn.ReLU(), 
            nn.Conv2d(in_channels=hidden_units, 
                      out_channels=hidden_units, 
                      kernel_size=3, # how big is the square that's going over the image?
                      stride=1, # default
                      padding=0), # options = "valid" (no padding) or "same" (output has same shape as input) or int for specific number            
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, 
                      out_channels=hidden_units, 
                      kernel_size=3, # how big is the square that's going over the image?
                      stride=1, # default
                      padding=0), # options = "valid" (no padding) or "same" (output has same shape as input) or int for specific number 
            nn.ReLU(), 
            nn.Conv2d(in_channels=hidden_units, 
                      out_channels=hidden_units, 
                      kernel_size=3, # how big is the square that's going over the image?
                      stride=1, # default
                      padding=0), # options = "valid" (no padding) or "same" (output has same shape as input) or int for specific number            
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            # Converts any H × W feature map into 1 × 1
            nn.AdaptiveAvgPool2d((4, 4)), # can experiment with this later
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*4*4,  # i dont need the trick anymore. I am using adaptive pooling
                      out_features=output_shape)
        )
        
    def forward(self,x):
        #x = self.conv_block_1(x)
        #print(f"Output shape of conv_block_1: {x.shape}")
        #print(x.shape)
        #x = self.conv_block_2(x)
        #print(f"Output shape of conv_block_2: {x.shape}")
        #print(x.shape)
        #x=self.classifier(x)
        #print(f"Output classifier: {x.shape}")
        #return x    # you can use operation fusion to speed this up  self.classifier(self.conv_block_2(self.conv_block_1(x)))
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

In [ ]:
#%%writefile -a modular/model_builder.py
from torch import nn
class TinyVGG(nn.Module):
    """
    TinyVGG-style convolutional neural network for image classification.
    Model architecture copying TinyVGG from: 
    https://poloclub.github.io/cnn-explainer/
    """
    def __init__(self,input_shape: int,hidden_units: int,output_shape: int,pool_output_size: int = 4,dropout: float = 0.0,padding: int = 0):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3, # how big is the square that's going over the image?
                stride=1, # default
                padding=padding), # options = "valid" (no padding) or "same" (output has same shape as input) or int for specific number 
            nn.ReLU(),
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=padding),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=padding),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=padding),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.classifier = nn.Sequential(
            # Converts any H × W feature map into 1 × 1
            nn.AdaptiveAvgPool2d((pool_output_size, pool_output_size)), # can experiment with this later
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(in_features=hidden_units * pool_output_size * pool_output_size,# i dont need the trick anymore. I am using adaptive pooling
                out_features=output_shape)
        )

    def forward(self, x):
        return self.classifier(self.conv_block_2(self.conv_block_1(x))
        )

In [ ]:
from modular.model_builder import TinyVGG
torch.manual_seed(42)
model_0_0_tinyVGG = TinyVGG(input_shape=3, hidden_units=10, output_shape=len(class_names)).to(device)
model_0_0_tinyVGG

#### Try a forward pass on a single image (to test the model)

In [ ]:
image_batch, label_batch =next(iter(train_dataloader_5))

In [ ]:
model_0_0_tinyVGG(image_batch.to(device))

#### Use torchinfo to get an idea of the shapes going through our model

In [ ]:
from torchinfo import summary
summary(model_0_0_tinyVGG, input_size=[1, 3, 224, 224]) # do a test pass through of an example input siz

# 5 Experiments

## 5.0 model_0_1_tinyVGG on 5% of the data set. No Augmentation

| Class       | Full train | 5% subset |
| ----------- | ---------: | --------: |
| class4      |         61 |         3 |
| **class5**  |      **8** |     **0** |
| class8      |        102 |         5 |
| class11     |        144 |         7 |
| class14     |        144 |         7 |
| **class15** |      **8** |     **0** |
| class16     |         41 |         2 |
| class17     |         82 |         4 |

Above is the issue with the class imbalance. there are some classes that have no pictures in them. this might cause issue doing the experimentation plan
experiment heavily on 5% → take winners to 20% → then full dataset

### 5.0.0 Create transform and dataloader with no data augmentation

In [ ]:
from torchvision import transforms
IMG_SIZE=224
simple_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor() # use ToTensor() last to get everything between 0 & 1
])


In [ ]:
train_subset_5_dataset

In [ ]:
subset_indices_5 = train_subset_5_dataset.indices

In [ ]:
# Full dataset, but with simple transform
train_dataset_simple = RockClassificationDataset(
    root_dir=train_dir,
    image_transform=simple_transform
)

# Apply the SAME 5% indices only to the train
train_subset_5_simple = torch.utils.data.Subset(
    train_dataset_simple,
    subset_indices_5
)



val_dataset_5_simple = RockClassificationDataset(
    root_dir=val_dir,
    image_transform=simple_transform
)

In [ ]:
BATCH_SIZE = 32

# Turn datasets into iterables (batches)
train_dataloader_5_simple = DataLoader(train_subset_5_simple, 
    batch_size=BATCH_SIZE, 
    num_workers=os.cpu_count(),                          
    shuffle=True 
)
val_dataloader_5_simple = DataLoader(val_dataset_5_simple, 
    batch_size=BATCH_SIZE, 
    num_workers=os.cpu_count(),                           
    shuffle=False 
)

In [ ]:
torch.manual_seed(42)
model_0_0_tinyVGG_results = TinyVGG(input_shape=3, 
    hidden_units=10, 
    output_shape=len(class_names)).to(device)
model_0_0_tinyVGG_results

In [ ]:
#%%writefile modular/engine.py
from timeit import default_timer as timer
import torch
from torch import nn

from .train_test import train_test
import wandb
def run_experiment(
    model_class,
    model_kwargs,
    train_dataloader,
    val_dataloader,
    train_metrics,
    test_metrics,
    device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs=None,

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs=None,
    run_name=None,

    seed=42
):
    """
    Creates, trains, and evaluates a PyTorch model for one experiment.
    Parameters
    ----------
    model_class : torch.nn.Module
        Model class to instantiate, e.g. TinyVGG.
    
    model_kwargs : dict
        Keyword arguments passed to the model class when creating
        a fresh model instance.
    
    train_dataloader : DataLoader
        DataLoader containing the training dataset.
    
    val_dataloader : DataLoader
        DataLoader containing the validation dataset.
    
    train_metrics :
        TorchMetrics collection used to calculate training metrics.
    
    test_metrics :
        TorchMetrics collection used to calculate validation metrics.
    
    device : torch.device
        Device used for training, e.g. "cuda" or "cpu".
    
    epochs : int
        Number of training epochs. Default is 5.
    
    optimizer_class : torch.optim.Optimizer
        PyTorch optimizer class used to train the model.
        Default is torch.optim.Adam.
    
    optimizer_kwargs : dict or None
        Keyword arguments passed to the optimizer.
        Can include values such as learning rate, weight decay,
        momentum, etc.
    
        Example:
            {
                "lr": 0.001,
                "weight_decay": 0.01
            }
    
    loss_class : torch.nn.Module
        PyTorch loss function class used to create the loss function.
        Default is nn.CrossEntropyLoss.
    
    loss_kwargs : dict or None
        Keyword arguments passed to the loss function.
    
        Can be used for options such as class weighting or
        label smoothing.
    
        Example for weighted cross entropy:
            {
                "weight": class_weights.to(device)
            }
    
        Example for label smoothing:
            {
                "label_smoothing": 0.1
            }
    
    seed : int
        Random seed used for reproducibility. Default is 42.
    
    Returns
    -------
    model : torch.nn.Module
        Trained model.
    
    results : dict
        Dictionary containing training and validation loss,
        metrics, and epoch training times.
    
    total_training_time : float
        Total training time for the experiment in seconds.
    """

    # ------------------------------------------
    #0. Create W&B experiment
    # ------------------------------------------

    wandb.init(
        project="rock-classification",
        name=run_name,
        config={
            "model": model_class.__name__,
            "epochs": epochs,
            "optimizer": optimizer_class.__name__,
            "loss": loss_class.__name__,
    
            "learning_rate": optimizer_kwargs.get("lr"),
            "batch_size": train_dataloader.batch_size,
    
            **model_kwargs,
            **loss_kwargs
        }
    )
    
    # -------------------------------------------------
    # 1. Set random seeds
    # -------------------------------------------------
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    # -------------------------------------------------
    # 2. Create a fresh model
    # -------------------------------------------------
    model = model_class(**model_kwargs).to(device)
    # -------------------------------------------------
    # 3. Create loss function
    # -------------------------------------------------
    if loss_kwargs is None:
        loss_kwargs = {}
    
    loss_fn = loss_class(**loss_kwargs)
    # -------------------------------------------------
    # 4. Create optimizer
    # -------------------------------------------------
    if optimizer_kwargs is None:
        optimizer_kwargs = {"lr": 0.001}
    
    optimizer = optimizer_class(
        model.parameters(),
        **optimizer_kwargs
    )
    # -------------------------------------------------
    # 5. Print experiment information
    # -------------------------------------------------
    print(f"Model: {model.__class__.__name__}")
    print(f"Epochs: {epochs}")
    print(f"Loss function: {loss_fn.__class__.__name__}")
    print(f"Optimizer: {optimizer.__class__.__name__}")
    print(f"Optimizer settings: {optimizer_kwargs}")
    print(f"Train batch size: {train_dataloader.batch_size}")
    print(f"Validation batch size: {val_dataloader.batch_size}")
    print("-" * 50)
    # -------------------------------------------------
    # 6. Start timer
    # -------------------------------------------------
    start_time = timer()
    # -------------------------------------------------
    # 7. Train model
    # -------------------------------------------------
    results = train_test(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=val_dataloader,
        optimizer=optimizer,
        loss_fn=loss_fn,
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        device=device,
        epochs=epochs
    )
    # -------------------------------------------------
    # 8. Stop timer
    # -------------------------------------------------
    end_time = timer()
    total_training_time = end_time - start_time
    print("-" * 50)
    print(f"Total training time: {total_training_time:.3f} seconds")
    
    # 9. Finish W&B experiment
    wandb.finish()

    return model, results, total_training_time

In [ ]:
from modular.engine import run_experiment
from modular.train_test import train_test
model_0_0_tinyVGG, model_0_0_tinyVGG_results, training_time = run_experiment(
    model_class=TinyVGG,

    model_kwargs={
        "input_shape": 3,
        "hidden_units": 10,
        "output_shape": len(class_names),
        "padding": 0,
        "dropout": 0.0
    },

    train_dataloader=train_dataloader_5_simple,
    val_dataloader=val_dataloader_5_simple,

    train_metrics=train_metrics,
    test_metrics=test_metrics,

    device=device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={
        "lr": 0.001
    },

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs={}
)

## 5.1 model_0_1_tinyVGG on 5% of the data Augmentation

This section might be out of order because I used the transformation that was developed in section 1. that is why u don't see the creation of the transformation and the dataloaders in here. Also for all other experiments that uses this augmentation, you are not going to see it

### 5.1.1 Create train & test loop functions

In [ ]:
loss_fn = nn.CrossEntropyLoss() # this is also called "criterion"/"cost function" in some places
optimizer = torch.optim.SGD(params=model_0_0_tinyVGG.parameters(), lr=0.1)

In [ ]:
model_0_1_tinyVGG, model_0_1_tinyVGG_results, training_time = run_experiment(
    model_class=TinyVGG,

    model_kwargs={
        "input_shape": 3,
        "hidden_units": 10,
        "output_shape": len(class_names),
        "padding": 0,
        "dropout": 0.0
    },

    train_dataloader=train_dataloader_5,
    val_dataloader=val_dataloader_5,

    train_metrics=train_metrics,
    test_metrics=test_metrics,

    device=device,
    epochs=5,

    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={
        "lr": 0.001
    },

    loss_class=nn.CrossEntropyLoss,
    loss_kwargs={}
)

In [ ]:
model_0_0_tinyVGG_results_df = pd.DataFrame(model_0_0_tinyVGG_results)
model_0_0_tinyVGG_results_df

In [ ]:
model_0_1_tinyVGG_results_df = pd.DataFrame(model_0_1_tinyVGG_results)
model_0_1_tinyVGG_results_df

In [ ]:
from modular.utility import summarize_experiment

In [ ]:
experiment_0_0 = summarize_experiment(
    results_df=model_0_0_tinyVGG_results_df,
    model=model_0_0_tinyVGG,
    experiment_name="0_0",
    model_name="TinyVGG",
    img_size=224,
    loss_fn="CrossEntropyLoss",
    notes="Baseline TinyVGG"
)

In [ ]:
experiment_0_1 =summarize_experiment(
    results_df=model_0_1_tinyVGG_results_df,
    model=model_0_1_tinyVGG,
    experiment_name="0_1",
    model_name="TinyVGG",
    img_size=224,
    loss_fn="CrossEntropyLoss",
    notes="augmented TinyVGG"
)

In [ ]:
experiment_summary_df = pd.DataFrame([
    experiment_0_0,
    experiment_0_1
])
experiment_summary_df

### 5.1.2 Plot the loss curve of Model 0